# 01 Load Dataset

Phase 3: AI Risk Model Foundation

This notebook loads exported CSV snapshots for the baseline risk-model pipeline. It records reproducibility metadata, validates expected session/attempt files, and checks for PII-sensitive columns before later data-quality and feature-engineering notebooks run.

Research constraints:

- BS-SA/BSSA remains the main framework.
- 2C3L is a scoring rubric / assessment mechanism under the Assessment Layer.
- Phase 3 baseline excludes post-submission 2C3L scores from model inputs to avoid leakage.
- No deep learning models are used in Phase 3.


## 1. Configuration

Place exported CSV files in `notebooks/data/raw/`.

Recommended filenames:

- `session_YYYYMMDD_batchXXX.csv`
- `attempt_YYYYMMDD_batchXXX.csv`


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import pandas as pd

RAW_DIR = Path("data/raw")
METADATA_PATH = Path("data/snapshot_metadata.json")

SESSION_CSV = None  # Example: RAW_DIR / "session_20260710_batch001.csv"
ATTEMPT_CSV = None  # Example: RAW_DIR / "attempt_20260710_batch001.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)


## 2. Locate CSV snapshots

If `SESSION_CSV` or `ATTEMPT_CSV` is left as `None`, the notebook tries to find the newest matching file in `data/raw/`.


In [ ]:
def newest_matching(pattern: str) -> Path | None:
    files = sorted(RAW_DIR.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

session_path = Path(SESSION_CSV) if SESSION_CSV else newest_matching("session_*.csv")
attempt_path = Path(ATTEMPT_CSV) if ATTEMPT_CSV else newest_matching("attempt_*.csv")

if session_path is None:
    raise FileNotFoundError("No session CSV found. Put session_YYYYMMDD_batchXXX.csv in notebooks/data/raw/.")

if attempt_path is None:
    raise FileNotFoundError("No attempt CSV found. Put attempt_YYYYMMDD_batchXXX.csv in notebooks/data/raw/.")

print(f"Session CSV: {session_path}")
print(f"Attempt CSV: {attempt_path}")


## 3. Load datasets


In [ ]:
session_df = pd.read_csv(session_path)
attempt_df = pd.read_csv(attempt_path)

print("Session shape:", session_df.shape)
print("Attempt shape:", attempt_df.shape)

display(session_df.head())
display(attempt_df.head())


## 4. Validate minimum required columns

The exact export schema can evolve, but Phase 3 requires these minimum fields before feature engineering can proceed.


In [ ]:
REQUIRED_SESSION_COLUMNS = {
    "batch_id",
    "task_id",
    "total_run_count",
    "total_attempt_count",
    "time_to_first_correct_sec",
}

REQUIRED_ATTEMPT_COLUMNS = {
    "task_id",
    "attempt_type",
    "is_correct",
}

def validate_required_columns(df: pd.DataFrame, required: set[str], name: str) -> None:
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")
    print(f"{name}: required columns present")

validate_required_columns(session_df, REQUIRED_SESSION_COLUMNS, "session_df")
validate_required_columns(attempt_df, REQUIRED_ATTEMPT_COLUMNS, "attempt_df")


## 5. PII-sensitive column check

Research snapshots should use anonymized identifiers. Do not commit raw CSV files.


In [ ]:
PII_SENSITIVE_COLUMNS = {
    "auth_user_id",
    "email",
    "student_id",
    "teacher_id",
    "display_name",
    "full_name",
    "first_name",
    "last_name",
}

def find_sensitive_columns(df: pd.DataFrame) -> list[str]:
    lowered = {col.lower(): col for col in df.columns}
    return [lowered[col] for col in PII_SENSITIVE_COLUMNS if col in lowered]

session_sensitive = find_sensitive_columns(session_df)
attempt_sensitive = find_sensitive_columns(attempt_df)

print("Session sensitive columns:", session_sensitive)
print("Attempt sensitive columns:", attempt_sensitive)

if session_sensitive or attempt_sensitive:
    print("WARNING: Sensitive columns detected. Drop or anonymize them before feature engineering/exporting processed data.")


## 6. Snapshot metadata

This metadata supports reproducibility without committing raw CSV data.


In [ ]:
metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "session_csv": str(session_path),
    "attempt_csv": str(attempt_path),
    "session_rows": int(len(session_df)),
    "session_columns": list(session_df.columns),
    "attempt_rows": int(len(attempt_df)),
    "attempt_columns": list(attempt_df.columns),
    "pii_sensitive_columns": {
        "session": session_sensitive,
        "attempt": attempt_sensitive,
    },
}

METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
METADATA_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(metadata, indent=2, ensure_ascii=False))


## 7. Next step

Proceed to `02_data_quality_check.ipynb` after confirming:

- row counts are plausible
- required columns are present
- PII-sensitive columns are absent or intentionally anonymized
- the snapshot date and batch code are documented
